In [1]:
# pyright: reportMissingTypeStubs=false, reportUnknownMemberType=false, reportUnknownArgumentType=false, reportUnknownVariableType=false, reportUnnecessaryCast=false
from typing import Any, TypeAlias, cast

import sympy as sp

SymValue: TypeAlias = Any
SymMatrix: TypeAlias = Any

# ---------- basic vector helpers ----------

def dot(a: SymMatrix, b: SymMatrix) -> SymValue:
    return cast(SymValue, (a.T * b)[0])

def norm(v: SymMatrix) -> SymValue:
    return cast(SymValue, sp.sqrt(dot(v, v)))

def unit(v: SymMatrix) -> SymMatrix:
    return cast(SymMatrix, sp.simplify(v / norm(v)))

def pretty_vec(v: SymMatrix) -> list[SymValue]:
    return [cast(SymValue, sp.simplify(x)) for x in list(v)]


# ---------- model setup ----------

# near plane normal direction, not normalized
m: SymMatrix = sp.Matrix([1, 2, 2])

# near plane center
p: SymMatrix = sp.Matrix([1, 2, 2])

# half-width vector on near plane
R: SymMatrix = sp.Matrix([0, -4, 4])

# half-height vector on near plane
U: SymMatrix = sp.Matrix([-4, 1, 1])

# unit normal from origin toward camera
n_hat: SymMatrix = unit(m)

# camera forward direction, camera looks toward origin
forward: SymMatrix = -n_hat


# ---------- validation ----------

print("m dot R =", sp.simplify(dot(m, R)))
print("m dot U =", sp.simplify(dot(m, U)))
print("R dot U =", sp.simplify(dot(R, U)))

print("|m| =", norm(m))
print("|R| =", norm(R))
print("|U| =", norm(U))

aspect = sp.simplify(norm(R) / norm(U))
print("aspect =", aspect)


# ---------- near corners ----------

corners: list[SymMatrix] = [
    p + R + U,
    p + R - U,
    p - R + U,
    p - R - U,
]

print("\nnear corners:")
for c in corners:
    print(pretty_vec(c))


# ---------- near plane equation ----------

x, y, z = cast(tuple[SymValue, SymValue, SymValue], sp.symbols("x y z", real=True))
X: SymMatrix = sp.Matrix([x, y, z])

plane_lhs: SymValue = dot(m, X)
plane_rhs: SymValue = dot(m, p)

print("\nnear plane:")
print(sp.Eq(plane_lhs, plane_rhs))

print("\ncheck corners on near plane:")
for q in corners:
    print(sp.simplify(dot(m, q)))


# ---------- camera position from FOV ----------

def camera_from_fov_deg(fov_deg: int) -> tuple[SymValue, SymMatrix, SymMatrix]:
    fov: SymValue = sp.rad(fov_deg)
    half_height: SymValue = norm(U)

    near: SymValue = sp.simplify(half_height / sp.tan(fov / 2))
    near = cast(SymValue, sp.radsimp(near))

    camera_pos: SymMatrix = sp.simplify(p + near * n_hat)
    camera_pos = sp.Matrix([sp.radsimp(v) for v in camera_pos])

    check_near_center: SymMatrix = sp.simplify(camera_pos + near * forward)

    return near, camera_pos, check_near_center


for deg in [60, 45, 30]:
    near, camera_pos, check = camera_from_fov_deg(deg)

    print(f"\nFOV = {deg} degrees")
    print("near =", near)
    print("camera =", pretty_vec(camera_pos))
    print("camera + near * forward =", pretty_vec(check))

m dot R = 0
m dot U = 0
R dot U = 0
|m| = 3
|R| = 4*sqrt(2)
|U| = 3*sqrt(2)
aspect = 4/3

near corners:
[-3, -1, 7]
[5, -3, 5]
[-3, 7, -1]
[5, 5, -3]

near plane:
Eq(x + 2*y + 2*z, 9)

check corners on near plane:
9
9
9
9

FOV = 60 degrees
near = 3*sqrt(6)
camera = [1 + sqrt(6), 2 + 2*sqrt(6), 2 + 2*sqrt(6)]
camera + near * forward = [1, 2, 2]

FOV = 45 degrees
near = 3*sqrt(2) + 6
camera = [sqrt(2) + 3, 2*sqrt(2) + 6, 2*sqrt(2) + 6]
camera + near * forward = [1, 2, 2]

FOV = 30 degrees
near = 3*sqrt(6) + 6*sqrt(2)
camera = [1 + sqrt(6) + 2*sqrt(2), 2 + 2*sqrt(6) + 4*sqrt(2), 2 + 2*sqrt(6) + 4*sqrt(2)]
camera + near * forward = [1, 2, 2]


In [1]:
import numpy as np
import numpy.typing as npt

HeightData = npt.NDArray[np.float64]


class QuadtreeNode:
    def __init__(
        self,
        x: int,
        y: int,
        size: int,
        depth: int,
        height_data: HeightData,
        max_depth: int,
        threshold: float,
    ) -> None:
        self.x: int = x                # 区域左上角的x坐标
        self.y: int = y                # 区域左上角的y坐标
        self.size: int = size          # 区域的尺寸（假设为正方形）
        self.depth: int = depth        # 当前节点的深度
        self.height_data: HeightData = height_data  # 区域内的高度数据（二维数组）
        self.max_depth: int = max_depth      # 最大递归深度
        self.threshold: float = threshold      # 误差阈值
        self.children: list[QuadtreeNode] = []        # 子节点列表
        self.is_leaf: bool = False      # 是否为叶子节点

        self.subdivide()          # 开始细分

    def subdivide(self) -> None:
        max_height = float(self.height_data.max())
        min_height = float(self.height_data.min())
        delta = max_height - min_height

        # 判断是否需要细分
        if delta > self.threshold and self.depth < self.max_depth and self.size > 1:
            half_size = self.size // 2

            # 获取子区域的高度数据
            data00 = self.height_data[0:half_size, 0:half_size]                # 左上
            data01 = self.height_data[0:half_size, half_size:self.size]        # 右上
            data10 = self.height_data[half_size:self.size, 0:half_size]        # 左下
            data11 = self.height_data[half_size:self.size, half_size:self.size]# 右下

            # 创建四个子节点
            self.children.append(QuadtreeNode(self.x, self.y, half_size, self.depth+1, data00, self.max_depth, self.threshold))
            self.children.append(QuadtreeNode(self.x + half_size, self.y, half_size, self.depth+1, data01, self.max_depth, self.threshold))
            self.children.append(QuadtreeNode(self.x, self.y + half_size, half_size, self.depth+1, data10, self.max_depth, self.threshold))
            self.children.append(QuadtreeNode(self.x + half_size, self.y + half_size, half_size, self.depth+1, data11, self.max_depth, self.threshold))
        else:
            self.is_leaf = True  # 不再细分，标记为叶子节点

# 假设height_map是一个二维的numpy数组，代表高度图
height_map: HeightData = (np.random.rand(128, 128) * 255).astype(np.float64)  # 生成一个随机高度图

# 创建四叉树
root_node = QuadtreeNode(x=0, y=0, size=128, depth=0, height_data=height_map, max_depth=6, threshold=10.0)